# 06 — Inspección visual del overlay multi-clase

Validación **visual** de la tarea `segmentation_overlay` (`show_overlay`).
Extrae un frame real, corre `detect_classes_in_frame` y dibuja el overlay con
leyenda.

> Requiere **GPU + pesos de SAM3** (la inferencia en CPU es inviable). Ejecutar
> en RunPod / contenedor con GPU.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

from src.core import detect_classes_in_frame, extract_frames, show_overlay
from src.utils import PROJECT_ROOT, get_abs_path

In [ ]:
# Resolver dataset_dir desde la config y localizar un .MOV (recursivo).
load_dotenv()
config_filename = os.getenv("CONFIG_FILENAME", "00_testing_config.json").strip()
config = json.loads(get_abs_path(f"configs/{config_filename}").read_text())
dataset_rel = config["working_dirs"]["dataset_dir"]

movs = sorted(
    {*(PROJECT_ROOT / dataset_rel).rglob("*.MOV"), *(PROJECT_ROOT / dataset_rel).rglob("*.mov")}
)
video = movs[0].relative_to(PROJECT_ROOT)
print(f"Video: {video}  ({len(movs)} .MOV encontrados)")

In [ ]:
# Extraer un frame (el central de la cuota).
frames = extract_frames(video)
frame = frames[len(frames) // 2]
print(f"Frame: {frame.shape}  dtype={frame.dtype}")

In [ ]:
# Detectar todas las clases (SAM3, requiere GPU) y mostrar el overlay con leyenda.
detections_by_class = detect_classes_in_frame(frame)
for name, dets in detections_by_class.items():
    print(f"  {name:12s} dets={len(dets)}")

show_overlay(frame, detections_by_class)